# Chapter 16: Operator Overloading
*Fluent Python, 2nd Edition -- Luciano Ramalho*

This notebook distills the key concepts from Chapter 16: how to overload unary, arithmetic, comparison, and augmented-assignment operators in Python, with fully self-contained runnable examples.

---

## 1. Operator Overloading 101 -- The Rules

Python lets user-defined types work with operators like `+`, `*`, `@`, `==`, etc.
But it imposes **three guardrails**:

1. You **cannot** change the meaning of operators for built-in types.
2. You **cannot** create new operators -- only overload existing ones.
3. A few operators **cannot** be overloaded: `is`, `and`, `or`, `not` (but bitwise `&`, `|`, `~` can).

Best practices:
- **Always return a new object** from non-augmented operators (never mutate `self`).
- **Return `NotImplemented`** (the singleton, NOT the exception) when you cannot handle an operand type.
- Use **duck typing** or **goose typing** (`isinstance` against ABCs) for type checks.

## 2. Unary Operators: `__neg__`, `__pos__`, `__abs__`, `__invert__`

| Operator | Method | Description |
|----------|--------------|-------------------------------|
| `-x`    | `__neg__`    | Arithmetic negation           |
| `+x`    | `__pos__`    | Arithmetic unary plus         |
| `~x`    | `__invert__` | Bitwise inverse               |
| `abs(x)`| `__abs__`    | Absolute value (returns scalar)|

**Key rule:** always return a *new* object.

In [ ]:
import math
import itertools
from array import array


class Vector:
    """Minimal Vector supporting unary operators."""
    typecode = "d"

    def __init__(self, components):
        self._components = array(self.typecode, components)

    def __iter__(self):
        return iter(self._components)

    def __len__(self):
        return len(self._components)

    def __repr__(self):
        components = ", ".join(f"{c!r}" for c in self._components)
        return f"Vector([{components}])"

    # --- Unary operators ---
    def __neg__(self):
        return Vector(-x for x in self)

    def __pos__(self):
        return Vector(self)           # new copy with same components

    def __abs__(self):
        return math.hypot(*self)

    # Note: __invert__ deliberately not implemented (meaningless for vectors)


v = Vector([1, -2, 3])
print(f"  v  = {v}")
print(f" -v  = {-v}")
print(f" +v  = {+v}")
print(f"|v|  = {abs(v):.4f}")

try:
    ~v
except TypeError as e:
    print(f" ~v  -> TypeError: {e}")

### Edge case: when `x != +x`

Two standard-library surprises where `+x` differs from `x`:
1. **`decimal.Decimal`** -- if precision changes between creation and `+x`.
2. **`collections.Counter`** -- `+counter` drops zero/negative tallies.

In [ ]:
import decimal
from collections import Counter

# --- Decimal precision change ---
ctx = decimal.getcontext()
ctx.prec = 40
one_third = decimal.Decimal("1") / decimal.Decimal("3")
print(f"precision=40: one_third = {one_third}")
print(f"one_third == +one_third ? {one_third == +one_third}")

ctx.prec = 28  # lower precision
print(f"precision=28: +one_third = {+one_third}")
print(f"one_third == +one_third ? {one_third == +one_third}")

# --- Counter drops zero/negative ---
ct = Counter("abracadabra")
ct["r"] = -3
ct["d"] = 0
print("")
print(f"Counter with negatives: {ct}")
print(f"+Counter (cleaned):     {+ct}")

## 3. Overloading `+` with `__add__` and `__radd__`

For the expression `a + b`, Python follows this dispatch algorithm:

1. Call `a.__add__(b)`. If it returns `NotImplemented` ...
2. Call `b.__radd__(a)`. If that also returns `NotImplemented` ...
3. Raise `TypeError`.

**`NotImplemented`** is a *singleton value* you *return*.
**`NotImplementedError`** is an *exception* you *raise*. Never confuse them.

For **commutative** operators, `__radd__` simply delegates to `__add__`.

In [ ]:
class Vector:
    typecode = "d"

    def __init__(self, components):
        self._components = array(self.typecode, components)

    def __iter__(self):
        return iter(self._components)

    def __len__(self):
        return len(self._components)

    def __repr__(self):
        components = ", ".join(f"{c!r}" for c in self._components)
        return f"Vector([{components}])"

    def __neg__(self):
        return Vector(-x for x in self)

    def __pos__(self):
        return Vector(self)

    def __abs__(self):
        return math.hypot(*self)

    # --- + operator ---
    def __add__(self, other):
        try:
            pairs = itertools.zip_longest(self, other, fillvalue=0.0)
            return Vector(a + b for a, b in pairs)
        except TypeError:
            return NotImplemented

    def __radd__(self, other):
        return self + other   # commutative: just delegate


v1 = Vector([3, 4, 5])
v2 = Vector([6, 7, 8])
print(f"v1 + v2          = {v1 + v2}")

# Works with tuples too (any iterable of numbers)
print(f"v1 + (10, 20, 30) = {v1 + (10, 20, 30)}")

# __radd__ lets the reversed order work
print(f"(10, 20, 30) + v1 = {(10, 20, 30) + v1}")

# Different lengths: shorter is zero-padded
v3 = Vector([1, 2])
print(f"v1 + v3          = {v1 + v3}")

# Incompatible type: graceful error via NotImplemented
try:
    v1 + "ABC"
except TypeError as e:
    print(f"v1 + 'ABC'       -> TypeError: {e}")

## 4. Overloading `*` for Scalar Multiplication

For scalar multiplication, we attempt to convert the operand to `float` (duck typing). If that fails, return `NotImplemented`.

In [ ]:
class Vector:
    typecode = "d"

    def __init__(self, components):
        self._components = array(self.typecode, components)

    def __iter__(self):
        return iter(self._components)

    def __len__(self):
        return len(self._components)

    def __repr__(self):
        components = ", ".join(f"{c!r}" for c in self._components)
        return f"Vector([{components}])"

    def __add__(self, other):
        try:
            pairs = itertools.zip_longest(self, other, fillvalue=0.0)
            return Vector(a + b for a, b in pairs)
        except TypeError:
            return NotImplemented

    def __radd__(self, other):
        return self + other

    # --- * operator (scalar multiplication) ---
    def __mul__(self, scalar):
        try:
            factor = float(scalar)
        except (TypeError, ValueError):   # float("hello") -> ValueError
            return NotImplemented
        return Vector(n * factor for n in self)

    def __rmul__(self, scalar):
        return self * scalar  # commutative


v1 = Vector([1.0, 2.0, 3.0])
print(f"v1 * 10       = {v1 * 10}")
print(f"11 * v1       = {11 * v1}")
print(f"v1 * True     = {v1 * True}")      # bool is subclass of int

from fractions import Fraction
print(f"v1 * 1/3      = {v1 * Fraction(1, 3)}")

try:
    v1 * "hello"
except TypeError as e:
    print(f"v1 * 'hello'  -> TypeError: {e}")

## 5. The `@` Infix Operator (Matrix/Dot Product) -- PEP 465

Since Python 3.5, `@` is an infix operator for matrix multiplication.
Special methods: `__matmul__`, `__rmatmul__`, `__imatmul__`.

This example uses **goose typing**: `isinstance` checks against `abc.Sized` and `abc.Iterable`. Any object with `__len__` and `__iter__` passes -- no subclassing/registration needed (thanks to `__subclasshook__`).

In [ ]:
from collections import abc


class Vector:
    typecode = "d"

    def __init__(self, components):
        self._components = array(self.typecode, components)

    def __iter__(self):
        return iter(self._components)

    def __len__(self):
        return len(self._components)

    def __repr__(self):
        components = ", ".join(f"{c!r}" for c in self._components)
        return f"Vector([{components}])"

    # --- @ operator (dot product) using goose typing ---
    def __matmul__(self, other):
        if isinstance(other, abc.Sized) and isinstance(other, abc.Iterable):
            if len(self) == len(other):
                return sum(a * b for a, b in zip(self, other))
            else:
                raise ValueError("@ requires vectors of equal length.")
        else:
            return NotImplemented

    def __rmatmul__(self, other):
        return self @ other


va = Vector([1, 2, 3])
vz = Vector([5, 6, 7])
print(f"va @ vz = {va @ vz}")          # 1*5 + 2*6 + 3*7 = 38.0
print(f"[10,20,30] @ vz = {[10, 20, 30] @ vz}")  # works with lists!

try:
    va @ 3
except TypeError as e:
    print(f"va @ 3 -> TypeError: {e}")

try:
    va @ Vector([1, 2])
except ValueError as e:
    print(f"va @ Vector([1,2]) -> ValueError: {e}")

## 6. Rich Comparison Operators

Rich comparison operators have **special dispatch rules**:

| Expression | Forward call | Reverse call | Fallback |
|------------|-------------|-------------|----------|
| `a == b`   | `a.__eq__(b)`  | `b.__eq__(a)`  | `id(a) == id(b)` |
| `a != b`   | `a.__ne__(b)`  | `b.__ne__(a)`  | `not (a == b)` |
| `a > b`    | `a.__gt__(b)`  | `b.__lt__(a)`  | `TypeError` |
| `a < b`    | `a.__lt__(b)`  | `b.__gt__(a)`  | `TypeError` |

Key differences from arithmetic operators:
- **Same method** is used for forward and reverse in `==`/`!=` (just swapped args).
- **`==` never raises** -- falls back to identity check.
- **`!=`** automatically negates `==` (via inherited `object.__ne__`).
- Ordering operators (`>`, `<`, `>=`, `<=`) **do raise** `TypeError` if unresolved.

In [ ]:
class Vector:
    typecode = "d"

    def __init__(self, components):
        self._components = array(self.typecode, components)

    def __iter__(self):
        return iter(self._components)

    def __len__(self):
        return len(self._components)

    def __repr__(self):
        components = ", ".join(f"{c!r}" for c in self._components)
        return f"Vector([{components}])"

    # --- Improved __eq__: only compare with Vector instances ---
    def __eq__(self, other):
        if isinstance(other, Vector):
            return len(self) == len(other) and all(
                a == b for a, b in zip(self, other)
            )
        else:
            return NotImplemented     # let the other side try


va = Vector([1.0, 2.0, 3.0])
vb = Vector(range(1, 4))
print(f"va == vb (same components)    : {va == vb}")       # True

# Tuple comparison: returns False (not equal by type)
t3 = (1, 2, 3)
print(f"va == (1, 2, 3) (diff type)   : {va == t3}")       # False

# != is auto-generated from __eq__
print(f"va != (1, 2, 3)               : {va != t3}")       # True
print(f"va != vb                      : {va != vb}")       # False

## 7. Augmented Assignment Operators (`__iadd__`, `__imul__`, etc.)

Without `__iadd__`, `a += b` desugars to `a = a.__add__(b)` -- **creates a new object**.
This is correct for **immutable** types.

For **mutable** types, implement `__iadd__` to modify `self` **in-place** and **return `self`**.

Key insight: `+=` can be **more liberal** than `+` in what it accepts on the right side.
Example: `list + x` requires `x` to be a list, but `list += x` accepts any iterable.

In [ ]:
import random


class BingoCage:
    """Simple bingo cage that picks items randomly."""

    def __init__(self, items):
        self._items = list(items)
        random.shuffle(self._items)

    def pick(self):
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError("pick from empty BingoCage")

    def inspect(self):
        return tuple(sorted(self._items))

    def __repr__(self):
        return f"BingoCage({self.inspect()})"


class AddableBingoCage(BingoCage):
    """BingoCage with + and += support."""

    def __add__(self, other):
        if isinstance(other, AddableBingoCage):
            return AddableBingoCage(self.inspect() + other.inspect())
        else:
            return NotImplemented

    def __iadd__(self, other):
        if isinstance(other, AddableBingoCage):
            other_iterable = other.inspect()
        else:
            try:
                other_iterable = iter(other)
            except TypeError:
                msg = "right operand in += must be AddableBingoCage or an iterable"
                raise TypeError(msg)
        self._items.extend(other_iterable)
        random.shuffle(self._items)
        return self   # MUST return self for in-place operators


# Demonstrate + creates NEW object
globe = AddableBingoCage("AEIOU")
globe2 = AddableBingoCage("XYZ")
globe3 = globe + globe2
print(f"globe  = {globe}")
print(f"globe2 = {globe2}")
print(f"globe3 = {globe3}")
print(f"globe3 is globe? {globe3 is globe}")   # False: new object

# Demonstrate += modifies IN-PLACE
original_id = id(globe)
globe += AddableBingoCage("MN")
print(f"")
print(f"After globe += AddableBingoCage('MN'): {globe}")
print(f"Same object? {id(globe) == original_id}")   # True: in-place

# += accepts any iterable (more liberal than +)
globe += ["P", "Q"]
print(f"After globe += ['P', 'Q']: {globe}")

# + rejects non-AddableBingoCage
try:
    globe + ["P", "Q"]
except TypeError as e:
    print(f"globe + list -> TypeError: {e}")

# += rejects non-iterables
try:
    globe += 42
except TypeError as e:
    print(f"globe += 42 -> TypeError: {e}")

### Immutable types: `+=` creates a new object automatically

If your immutable class has `__add__` but no `__iadd__`, Python handles `+=` by calling `__add__` and rebinding the variable.

In [ ]:
class Vector:
    typecode = "d"

    def __init__(self, components):
        self._components = array(self.typecode, components)

    def __iter__(self):
        return iter(self._components)

    def __len__(self):
        return len(self._components)

    def __repr__(self):
        components = ", ".join(f"{c!r}" for c in self._components)
        return f"Vector([{components}])"

    def __add__(self, other):
        try:
            pairs = itertools.zip_longest(self, other, fillvalue=0.0)
            return Vector(a + b for a, b in pairs)
        except TypeError:
            return NotImplemented

    def __radd__(self, other):
        return self + other

    def __mul__(self, scalar):
        try:
            factor = float(scalar)
        except (TypeError, ValueError):
            return NotImplemented
        return Vector(n * factor for n in self)

    def __rmul__(self, scalar):
        return self * scalar


v1 = Vector([1, 2, 3])
print(f"v1 = {v1}, id = {id(v1)}")

v1_alias = v1
v1 += Vector([4, 5, 6])
print(f"After v1 += Vector([4,5,6]): v1 = {v1}, id = {id(v1)}")
print(f"v1_alias (original unchanged) = {v1_alias}")
print(f"Same object? {v1 is v1_alias}")  # False: new object was created

v1 *= 11
print(f"After v1 *= 11: v1 = {v1}, id = {id(v1)}")

## Summary -- Operator Overloading Cheat Sheet

### Full Infix Operator Method Table

| Operator | Forward | Reverse | In-place | Description |
|----------|---------|---------|----------|-------------|
| `+` | `__add__` | `__radd__` | `__iadd__` | Addition / concatenation |
| `-` | `__sub__` | `__rsub__` | `__isub__` | Subtraction |
| `*` | `__mul__` | `__rmul__` | `__imul__` | Multiplication / repetition |
| `/` | `__truediv__` | `__rtruediv__` | `__itruediv__` | True division |
| `//` | `__floordiv__` | `__rfloordiv__` | `__ifloordiv__` | Floor division |
| `%` | `__mod__` | `__rmod__` | `__imod__` | Modulo |
| `**` | `__pow__` | `__rpow__` | `__ipow__` | Exponentiation |
| `@` | `__matmul__` | `__rmatmul__` | `__imatmul__` | Matrix multiplication |
| `&` | `__and__` | `__rand__` | `__iand__` | Bitwise AND |
| `\|` | `__or__` | `__ror__` | `__ior__` | Bitwise OR |
| `^` | `__xor__` | `__rxor__` | `__ixor__` | Bitwise XOR |
| `<<` | `__lshift__` | `__rlshift__` | `__ilshift__` | Shift left |
| `>>` | `__rshift__` | `__rrshift__` | `__irshift__` | Shift right |

### Key Takeaways

1. **Return `NotImplemented`** (not raise `NotImplementedError`) when you cannot handle an operand type.
2. **Never mutate operands** in `__add__`, `__mul__`, etc. -- always return new objects.
3. **`__radd__`** is tried when the left operand's `__add__` returns `NotImplemented`.
4. **Rich comparison** operators have special pairing (`__gt__` <-> `__lt__`) and `==` never raises.
5. **`__iadd__`** (augmented assignment) should mutate `self` in-place and return `self` for mutable types.
6. For immutable types, don't implement `__iadd__` -- Python falls back to `__add__` + rebind automatically.
7. `+=` can be **more permissive** than `+` for mutable types (e.g., `list += any_iterable`).